# Step Order — Sensitivity of Preprocessing Operator Order

This experiment tests whether the order of **outlier handling (OUT)** and **missing-value handling (MV)** changes downstream classification when all other components are held fixed.

Two paired conditions are evaluated:

```text
OUT-first:
OUT → MV → NORM → optional smoothing → optional filtering → optional feature selection

MV-first:
MV → OUT → completion with the same MV method
   → NORM → optional smoothing → optional filtering → optional feature selection
```

The data, pipeline configuration, cross-validation folds, feature extraction, classifier, and all later processing steps are identical between the paired conditions.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from scipy.signal import butter, filtfilt, medfilt
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.impute import KNNImputer
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import RobustScaler, StandardScaler

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
RESULTS_DIR = BASE_DIR / "results"
RECOMPUTED_DIR = BASE_DIR / "results_recomputed"

FS = 120
CV_SPLITS = 5
RANDOM_STATE = 42
IFOREST_CONTAMINATION = 0.02

RUN_EXPERIMENT = False

DATASETS = {
    "education": DATA_DIR / "education_timeseries.npz",
    "medical": DATA_DIR / "medical_timeseries.npz",
    "usability": DATA_DIR / "usability_timeseries.npz",
}

print("Working directory:", BASE_DIR)
print("RUN_EXPERIMENT:", RUN_EXPERIMENT)

## 1. Data

Each NPZ file contains `X_clean`, `X_noisy`, and the binary target `y`.

The paired order experiment uses `X_noisy`. The three synthetic scenarios contain 50 balanced samples each, with 7,200 time points and three channels per sample.

In [ ]:
def load_dataset(path):
    data = np.load(path)
    return data["X_clean"], data["X_noisy"], data["y"]

data_rows = []

for name, path in DATASETS.items():
    X_clean, X_noisy, y = load_dataset(path)

    data_rows.append({
        "dataset": name,
        "samples": len(y),
        "timepoints": X_noisy.shape[1],
        "channels": X_noisy.shape[2],
        "class_0": int((y == 0).sum()),
        "class_1": int((y == 1).sum()),
        "missing_rate": float(np.isnan(X_noisy).mean()),
    })

dataset_overview = pd.DataFrame(data_rows)
display(dataset_overview)

## 2. Fifteen pipeline configurations

The experiment evaluates 15 configurations (A–O). The order intervention changes only the relative position of OUT and MV; normalization and all optional later operations remain unchanged.

In [ ]:
PIPELINES = {
    "A_baseline": dict(
        outlier="none",
        mv="mean",
        norm="standard",
        smooth="none",
        butter=False,
        feat_limit=None,
    ),
    "B_robust": dict(
        outlier="iforest",
        mv="knn",
        norm="robust",
        smooth="none",
        butter=False,
        feat_limit=None,
    ),
    "C_signal": dict(
        outlier="iforest",
        mv="knn",
        norm="robust",
        smooth="fft",
        butter=True,
        feat_limit=None,
    ),
    "D_limit": dict(
        outlier="iforest",
        mv="knn",
        norm="robust",
        smooth="fft",
        butter=True,
        feat_limit=30,
    ),
    "E_clinical": dict(
        outlier="winsor",
        mv="locf",
        norm="standard",
        smooth="none",
        butter=False,
        feat_limit=None,
    ),
    "F_knn_only": dict(
        outlier="none",
        mv="knn",
        norm="robust",
        smooth="none",
        butter=False,
        feat_limit=None,
    ),
    "G_iqr_knn": dict(
        outlier="iqr_clip",
        mv="knn",
        norm="robust",
        smooth="none",
        butter=False,
        feat_limit=None,
    ),
    "H_if_mean": dict(
        outlier="iforest",
        mv="mean",
        norm="robust",
        smooth="none",
        butter=False,
        feat_limit=None,
    ),
    "I_locf_robust": dict(
        outlier="none",
        mv="locf",
        norm="robust",
        smooth="none",
        butter=False,
        feat_limit=None,
    ),
    "J_interp_std": dict(
        outlier="none",
        mv="linear_interp",
        norm="standard",
        smooth="none",
        butter=False,
        feat_limit=None,
    ),
    "K_norm_only": dict(
        outlier="none",
        mv="mean",
        norm="robust",
        smooth="none",
        butter=False,
        feat_limit=None,
    ),
    "L_fft_only": dict(
        outlier="iforest",
        mv="knn",
        norm="robust",
        smooth="fft",
        butter=False,
        feat_limit=None,
    ),
    "M_butter_only": dict(
        outlier="iforest",
        mv="knn",
        norm="robust",
        smooth="none",
        butter=True,
        feat_limit=None,
    ),
    "N_median_smooth": dict(
        outlier="none",
        mv="linear_interp",
        norm="robust",
        smooth="median",
        butter=False,
        feat_limit=None,
    ),
    "O_limit10": dict(
        outlier="iforest",
        mv="knn",
        norm="robust",
        smooth="fft",
        butter=True,
        feat_limit=10,
    ),
}

pipeline_table = (
    pd.DataFrame(PIPELINES)
    .T
    .reset_index()
    .rename(columns={"index": "pipeline"})
)

display(pipeline_table)

## 3. Missing-value handling

Mean imputation, KNN, LOCF, and linear interpolation are implemented per sample.

In the supplied arrays, incomplete time points are missing in all three channels simultaneously. For this specific missingness pattern, `KNNImputer` returns the feature-wise mean for a completely missing row. The implementation checks this condition and uses the numerically equivalent mean fast path. For any other missingness pattern it falls back to the standard `KNNImputer`.

In [ ]:
def interp_nan_1d(x):
    x = np.asarray(x, dtype=float).copy()
    index = np.arange(len(x))
    valid = ~np.isnan(x)

    if valid.sum() < 2:
        return np.nan_to_num(x, nan=0.0)

    return np.interp(
        index,
        index[valid],
        x[valid],
    )

def interp_nan_ts(X):
    X = np.asarray(X, dtype=float).copy()

    for channel in range(X.shape[1]):
        X[:, channel] = interp_nan_1d(
            X[:, channel]
        )

    return X

def mean_impute_ts(X):
    X = np.asarray(X, dtype=float).copy()

    for channel in range(X.shape[1]):
        values = X[:, channel]
        mean_value = np.nanmean(values)

        if not np.isfinite(mean_value):
            mean_value = 0.0

        values[np.isnan(values)] = mean_value
        X[:, channel] = values

    return X

def locf_impute_ts(X):
    X = np.asarray(X, dtype=float).copy()

    for channel in range(X.shape[1]):
        values = X[:, channel]
        last = np.nan

        for i in range(len(values)):
            if np.isnan(values[i]) and np.isfinite(last):
                values[i] = last
            elif np.isfinite(values[i]):
                last = values[i]

        mean_value = np.nanmean(values)

        if not np.isfinite(mean_value):
            mean_value = 0.0

        values[np.isnan(values)] = mean_value
        X[:, channel] = values

    return X

def linear_interp_ts(X):
    return interp_nan_ts(X)

def knn_impute_ts(X, k=5):
    X = np.asarray(X, dtype=float)
    missing = np.isnan(X)

    # Exact fast path for the supplied arrays:
    # every incomplete time point is missing in all channels.
    all_or_none_missing = np.all(
        (~missing.any(axis=1))
        | missing.all(axis=1)
    )

    if missing.any() and all_or_none_missing:
        return mean_impute_ts(X)

    imputer = KNNImputer(
        n_neighbors=k,
        weights="distance",
    )

    return imputer.fit_transform(X)

def apply_mv_batch(X, method):
    functions = {
        "mean": mean_impute_ts,
        "knn": knn_impute_ts,
        "locf": locf_impute_ts,
        "linear_interp": linear_interp_ts,
    }

    function = functions[method]

    return np.asarray([
        function(sample)
        for sample in X
    ])

## 4. Outlier handling

Isolation Forest is fitted only on complete training-fold time points. Detected time points are marked as missing.

Winsorisation and IQR clipping are deterministic per-sample operations.

For OUT-first, the configured MV method repairs original missingness and points marked by the outlier detector. For MV-first, the configured MV method first repairs original missingness; if Isolation Forest subsequently marks points as missing, the **same** MV method is applied again to those newly marked points before normalization.

In [ ]:
def winsorize_ts(
    X,
    low_q=0.01,
    high_q=0.99,
):
    X = np.asarray(X, dtype=float).copy()

    for channel in range(X.shape[1]):
        values = X[:, channel]
        valid = np.isfinite(values)

        if valid.sum() < 2:
            continue

        low, high = np.quantile(
            values[valid],
            [low_q, high_q],
        )

        values[valid] = np.clip(
            values[valid],
            low,
            high,
        )

        X[:, channel] = values

    return X

def iqr_clip_ts(X, k=1.5):
    X = np.asarray(X, dtype=float).copy()

    for channel in range(X.shape[1]):
        values = X[:, channel]
        valid = np.isfinite(values)

        if valid.sum() < 4:
            continue

        q1, q3 = np.quantile(
            values[valid],
            [0.25, 0.75],
        )

        iqr = q3 - q1

        values[valid] = np.clip(
            values[valid],
            q1 - k * iqr,
            q3 + k * iqr,
        )

        X[:, channel] = values

    return X

def apply_nonlearned_outlier_batch(
    X,
    method,
):
    functions = {
        "winsor": winsorize_ts,
        "iqr_clip": iqr_clip_ts,
    }

    function = functions[method]

    return np.asarray([
        function(sample)
        for sample in X
    ])

def fit_apply_iforest(
    X_train,
    X_test,
    contamination=IFOREST_CONTAMINATION,
    random_state=RANDOM_STATE,
):
    channels = X_train.shape[2]

    flat_train = (
        np.asarray(X_train, dtype=float)
        .reshape(-1, channels)
    )

    complete = ~np.isnan(
        flat_train
    ).any(axis=1)

    fit_data = flat_train[complete]

    if fit_data.shape[0] < 1000:
        return X_train.copy(), X_test.copy()

    n_fit = min(
        30000,
        fit_data.shape[0],
    )

    rng = np.random.default_rng(
        random_state
    )

    sample_index = rng.choice(
        fit_data.shape[0],
        size=n_fit,
        replace=False,
    )

    model = IsolationForest(
        contamination=contamination,
        random_state=random_state,
        n_jobs=-1,
    )

    model.fit(
        fit_data[sample_index]
    )

    def apply(X):
        output = np.asarray(
            X,
            dtype=float,
        ).copy()

        flat = output.reshape(
            -1,
            channels,
        )

        valid = ~np.isnan(
            flat
        ).any(axis=1)

        prediction = np.ones(
            len(flat),
            dtype=int,
        )

        prediction[valid] = model.predict(
            flat[valid]
        )

        flat[
            prediction == -1,
            :
        ] = np.nan

        return flat.reshape(
            output.shape
        )

    return apply(X_train), apply(X_test)

## 5. Normalization and optional signal processing

Standard or robust scaling is fitted on flattened training-fold time points and then applied to both train and validation data.

Optional later operations are FFT low-frequency reconstruction, a fourth-order 8 Hz Butterworth low-pass filter, and median smoothing with kernel size 5.

In [ ]:
def fit_time_scaler(
    X_train,
    kind,
):
    channels = X_train.shape[2]

    flat = X_train.reshape(
        -1,
        channels,
    )

    flat = flat[
        ~np.isnan(flat).any(axis=1)
    ]

    scaler = (
        RobustScaler()
        if kind == "robust"
        else StandardScaler()
    )

    return scaler.fit(flat)

def apply_time_scaler(
    X,
    scaler,
):
    shape = X.shape
    channels = shape[2]

    flat = np.asarray(
        X,
        dtype=float,
    ).reshape(
        -1,
        channels,
    )

    if np.isnan(flat).any():
        column_means = np.nanmean(
            flat,
            axis=0,
        )

        indices = np.where(
            np.isnan(flat)
        )

        flat[indices] = np.take(
            column_means,
            indices[1],
        )

    return scaler.transform(
        flat
    ).reshape(shape)

def fft_smooth_ts(
    X,
    keep_ratio=0.08,
):
    X = np.asarray(
        X,
        dtype=float,
    ).copy()

    for channel in range(
        X.shape[1]
    ):
        spectrum = np.fft.rfft(
            X[:, channel]
        )

        keep = max(
            2,
            int(
                len(spectrum)
                * keep_ratio
            ),
        )

        spectrum[keep:] = 0

        X[:, channel] = np.fft.irfft(
            spectrum,
            n=X.shape[0],
        )

    return X

def median_filter_ts(
    X,
    kernel_size=5,
):
    X = np.asarray(
        X,
        dtype=float,
    ).copy()

    for channel in range(
        X.shape[1]
    ):
        X[:, channel] = medfilt(
            X[:, channel],
            kernel_size=kernel_size,
        )

    return X

def butter_lowpass_ts(
    X,
    fs=FS,
    cutoff_hz=8.0,
    order=4,
):
    X = np.asarray(
        X,
        dtype=float,
    ).copy()

    b, a = butter(
        order,
        cutoff_hz / (0.5 * fs),
        btype="low",
    )

    for channel in range(
        X.shape[1]
    ):
        X[:, channel] = filtfilt(
            b,
            a,
            X[:, channel],
        )

    return X

## 6. Feature extraction and fixed classifier

One-second windows with 50% overlap are used.

For each window the feature vector contains channel-wise mean, standard deviation, minimum, maximum, gaze-displacement velocity statistics, and the dominant non-DC FFT magnitude for each channel. The window-level features are aggregated by their mean and standard deviation, producing 34 features per sample.

The classifier is fixed for every pipeline and both orders.

In [ ]:
def extract_features(
    X,
    fs=FS,
):
    X = np.asarray(
        X,
        dtype=float,
    )

    samples, timepoints, channels = X.shape

    window = fs
    step = fs // 2

    starts = np.arange(
        0,
        timepoints - window + 1,
        step,
    )

    index = (
        starts[:, None]
        + np.arange(window)[None, :]
    )

    segments = X[:, index, :]

    mean = segments.mean(axis=2)
    std = segments.std(axis=2)
    minimum = segments.min(axis=2)
    maximum = segments.max(axis=2)

    dx = np.diff(
        segments[:, :, :, 0],
        axis=2,
        prepend=segments[:, :, 0:1, 0],
    )

    dy = np.diff(
        segments[:, :, :, 1],
        axis=2,
        prepend=segments[:, :, 0:1, 1],
    )

    velocity = np.sqrt(
        dx ** 2 + dy ** 2
    )

    velocity_mean = (
        velocity
        .mean(axis=2)[..., None]
    )

    velocity_std = (
        velocity
        .std(axis=2)[..., None]
    )

    magnitude = np.abs(
        np.fft.rfft(
            segments,
            axis=2,
        )
    )

    dominant_fft = (
        magnitude[:, :, 1:, :]
        .max(axis=2)
    )

    window_features = np.concatenate(
        [
            mean,
            std,
            minimum,
            maximum,
            velocity_mean,
            velocity_std,
            dominant_fft,
        ],
        axis=2,
    )

    return np.concatenate(
        [
            window_features.mean(axis=1),
            window_features.std(axis=1),
        ],
        axis=1,
    )

def make_random_forest():
    return RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features="sqrt",
        bootstrap=True,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

## 7. Paired order evaluation

The same stratified five folds are used for both orders.

Raw performance is calculated with a temporary interpolation required only for numerical feature extraction. The paired preprocessing paths start from the unmodified `X_noisy` arrays.

Feature selection for D and O is performed within the training fold using Random-Forest feature importance.

In [ ]:
def score_features(
    X_train_features,
    X_test_features,
    y_train,
    y_test,
):
    model = make_random_forest()

    model.fit(
        X_train_features,
        y_train,
    )

    prediction = model.predict(
        X_test_features
    )

    probability = model.predict_proba(
        X_test_features
    )[:, 1]

    return {
        "accuracy": accuracy_score(
            y_test,
            prediction,
        ),
        "f1": f1_score(
            y_test,
            prediction,
        ),
        "auroc": roc_auc_score(
            y_test,
            probability,
        ),
    }

def repair_pair(
    X_train,
    X_test,
    config,
    order,
):
    outlier = config["outlier"]
    mv = config["mv"]

    if order == "out_first":
        if outlier == "iforest":
            X_train, X_test = fit_apply_iforest(
                X_train,
                X_test,
            )

        elif outlier == "winsor":
            X_train = apply_nonlearned_outlier_batch(
                X_train,
                "winsor",
            )
            X_test = apply_nonlearned_outlier_batch(
                X_test,
                "winsor",
            )

        elif outlier == "iqr_clip":
            X_train = apply_nonlearned_outlier_batch(
                X_train,
                "iqr_clip",
            )
            X_test = apply_nonlearned_outlier_batch(
                X_test,
                "iqr_clip",
            )

        X_train = apply_mv_batch(
            X_train,
            mv,
        )
        X_test = apply_mv_batch(
            X_test,
            mv,
        )

    elif order == "mv_first":
        X_train = apply_mv_batch(
            X_train,
            mv,
        )
        X_test = apply_mv_batch(
            X_test,
            mv,
        )

        if outlier == "iforest":
            X_train, X_test = fit_apply_iforest(
                X_train,
                X_test,
            )

            # Isolation Forest marks complete time points as NaN.
            # Reapply the SAME configured MV method only to obtain
            # a complete input for the later numerical stages.
            X_train = apply_mv_batch(
                X_train,
                mv,
            )
            X_test = apply_mv_batch(
                X_test,
                mv,
            )

        elif outlier == "winsor":
            X_train = apply_nonlearned_outlier_batch(
                X_train,
                "winsor",
            )
            X_test = apply_nonlearned_outlier_batch(
                X_test,
                "winsor",
            )

        elif outlier == "iqr_clip":
            X_train = apply_nonlearned_outlier_batch(
                X_train,
                "iqr_clip",
            )
            X_test = apply_nonlearned_outlier_batch(
                X_test,
                "iqr_clip",
            )

    else:
        raise ValueError(
            "order must be 'out_first' or 'mv_first'"
        )

    return X_train, X_test

def finish_pipeline(
    X_train,
    X_test,
    config,
):
    scaler = fit_time_scaler(
        X_train,
        config["norm"],
    )

    X_train = apply_time_scaler(
        X_train,
        scaler,
    )

    X_test = apply_time_scaler(
        X_test,
        scaler,
    )

    if config["smooth"] == "fft":
        X_train = np.asarray([
            fft_smooth_ts(sample)
            for sample in X_train
        ])
        X_test = np.asarray([
            fft_smooth_ts(sample)
            for sample in X_test
        ])

    elif config["smooth"] == "median":
        X_train = np.asarray([
            median_filter_ts(sample)
            for sample in X_train
        ])
        X_test = np.asarray([
            median_filter_ts(sample)
            for sample in X_test
        ])

    if config["butter"]:
        X_train = np.asarray([
            butter_lowpass_ts(sample)
            for sample in X_train
        ])
        X_test = np.asarray([
            butter_lowpass_ts(sample)
            for sample in X_test
        ])

    return X_train, X_test

def evaluate_one_pipeline(
    X_train,
    X_test,
    y_train,
    y_test,
    config,
    order,
):
    X_train, X_test = repair_pair(
        X_train,
        X_test,
        config,
        order,
    )

    X_train, X_test = finish_pipeline(
        X_train,
        X_test,
        config,
    )

    train_features = extract_features(
        X_train
    )

    test_features = extract_features(
        X_test
    )

    feature_limit = config[
        "feat_limit"
    ]

    if feature_limit is not None:
        selector = make_random_forest()

        selector.fit(
            train_features,
            y_train,
        )

        ranking = np.argsort(
            selector.feature_importances_
        )[::-1]

        selected = ranking[
            :int(feature_limit)
        ]

        train_features = train_features[
            :,
            selected,
        ]

        test_features = test_features[
            :,
            selected,
        ]

    return score_features(
        train_features,
        test_features,
        y_train,
        y_test,
    )

def run_dataset(
    dataset_name,
    path,
):
    _, X_noisy, y = load_dataset(
        path
    )

    splitter = StratifiedKFold(
        n_splits=CV_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    rows = []

    for fold, (
        train_index,
        test_index,
    ) in enumerate(
        splitter.split(
            X_noisy,
            y,
        ),
        start=1,
    ):
        X_train = X_noisy[
            train_index
        ].astype(float)

        X_test = X_noisy[
            test_index
        ].astype(float)

        y_train = y[
            train_index
        ]

        y_test = y[
            test_index
        ]

        for order in [
            "out_first",
            "mv_first",
        ]:
            for (
                pipeline_name,
                config,
            ) in PIPELINES.items():

                metrics = evaluate_one_pipeline(
                    X_train,
                    X_test,
                    y_train,
                    y_test,
                    config,
                    order,
                )

                rows.append({
                    "dataset": dataset_name,
                    "fold": fold,
                    "order": order,
                    "pipeline": pipeline_name,
                    **metrics,
                })

        print(
            f"{dataset_name}: fold {fold} finished"
        )

    return pd.DataFrame(rows)

def summarize_order_results(
    fold_results,
):
    summary = (
        fold_results
        .groupby(
            [
                "dataset",
                "pipeline",
                "order",
            ],
            as_index=False,
        )[
            [
                "accuracy",
                "f1",
                "auroc",
            ]
        ]
        .mean()
    )

    wide = summary.pivot(
        index=[
            "dataset",
            "pipeline",
        ],
        columns="order",
        values=[
            "accuracy",
            "f1",
            "auroc",
        ],
    )

    wide.columns = [
        f"{metric}_{order}"
        for metric, order in wide.columns
    ]

    wide = wide.reset_index()

    for metric in [
        "accuracy",
        "f1",
        "auroc",
    ]:
        wide[
            f"delta_{metric}_mv_minus_out"
        ] = (
            wide[
                f"{metric}_mv_first"
            ]
            - wide[
                f"{metric}_out_first"
            ]
        )

    return wide

## 8. Run or load the experiment

The supplied result files were produced with the paired design above.

Set `RUN_EXPERIMENT = True` to recompute the full experiment. A complete rerun is intentionally computationally expensive because it evaluates 15 configurations × 2 orders × 3 scenarios × 5 folds with a 300-tree Random Forest.

In [ ]:
if RUN_EXPERIMENT:
    result_frames = []

    for (
        dataset_name,
        path,
    ) in DATASETS.items():
        result_frames.append(
            run_dataset(
                dataset_name,
                path,
            )
        )

    fold_results = pd.concat(
        result_frames,
        ignore_index=True,
    )

    order_comparison = summarize_order_results(
        fold_results
    )

    RECOMPUTED_DIR.mkdir(
        exist_ok=True
    )

    fold_results.to_csv(
        RECOMPUTED_DIR
        / "fold_results.csv",
        index=False,
    )

    order_comparison.to_csv(
        RECOMPUTED_DIR
        / "order_comparison.csv",
        index=False,
    )

else:
    fold_results = pd.read_csv(
        RESULTS_DIR
        / "fold_results.csv"
    )

    order_comparison = pd.read_csv(
        RESULTS_DIR
        / "order_comparison.csv"
    )

print(
    "Fold-level rows:",
    len(fold_results),
)

display(
    order_comparison
)

## 9. Results

The raw baseline and best final configuration are summarized per scenario below.

The order comparison should be interpreted descriptively. The experiment is a synthetic sensitivity analysis and does not establish a universal ordering rule.

In [ ]:
dataset_summary = pd.read_csv(
    RESULTS_DIR
    / "dataset_summary.csv"
)

display(
    dataset_summary
)

order_sensitive_pipelines = [
    "B_robust",
    "C_signal",
    "D_limit",
    "E_clinical",
    "G_iqr_knn",
    "H_if_mean",
    "L_fft_only",
    "M_butter_only",
    "O_limit10",
]

order_sensitive_summary = (
    order_comparison[
        order_comparison[
            "pipeline"
        ].isin(
            order_sensitive_pipelines
        )
    ]
    .groupby(
        "dataset",
        as_index=False,
    )[
        "delta_accuracy_mv_minus_out"
    ]
    .mean()
)

display(
    order_sensitive_summary
)

### Interpretation

**Education.** Raw Accuracy is 0.68. The best full configuration is `J_interp_std` with 0.74 in both orders. Several Isolation-Forest configurations favour OUT-first, but the best overall configuration itself contains no outlier stage and is therefore order-invariant.

**Medical.** Raw Accuracy is 0.44. `N_median_smooth` reaches 0.50 in both orders. Order-sensitive configurations move in both directions, and the aggregate difference is small.

**Usability.** Raw Accuracy is 0.40. `O_limit10` reaches 0.58 with OUT-first and 0.54 with MV-first. Other configurations show both positive and negative order effects, so the results do not support a general MV-first advantage.

**Overall conclusion.** Operator order is part of the pipeline configuration space, but there is no universal preferred sequence. The observed effect depends on the concrete outlier method, missing-value method, and later signal-processing stages.